In [ ]:
import glob #mencari file mendalam

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True

    found_clinvar = glob.glob('/content/drive/MyDrive/**/variant_summary.txt.gz', recursive=True)
    if found_clinvar:
        RAW_CLINVAR = Path(found_clinvar[0])
        CLINVAR_DIR = RAW_CLINVAR.parent
        BASE_DIR = CLINVAR_DIR.parent
        print(f"[OK] File ClinVar BERHASIL ditemukan di:\n   -> {RAW_CLINVAR}")
    else:
        BASE_DIR = Path('/content/drive/MyDrive/skripsi/data')
        CLINVAR_DIR = BASE_DIR / 'ClinVar'
        RAW_CLINVAR = CLINVAR_DIR / 'variant_summary.txt.gz'

except ImportError:
    IN_COLAB = False
    BASE_DIR = Path('./data')
    CLINVAR_DIR = BASE_DIR / 'ClinVar'
    RAW_CLINVAR = CLINVAR_DIR / 'variant_summary.txt.gz'

CLEAN_CLINVAR = CLINVAR_DIR / 'clinvar_cleaned_grch38.csv'
LAMPIRAN_DIR = BASE_DIR / 'Lampiran_Skripsi'
LAMPIRAN_DIR.mkdir(parents=True, exist_ok=True)

print('=' * 60)
print('TAHAP 2: PRA-PEMROSESAN DATA CLINVAR')
print('=' * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] File ClinVar BERHASIL ditemukan di:
   -> /content/drive/MyDrive/Codingan Skripsi Muhammad Thariq Adzikra 2303113029/data/ClinVar/variant_summary.txt.gz
TAHAP 2: PRA-PEMROSESAN DATA CLINVAR


## 1. CEK FILE INPUT


In [ ]:
if not RAW_CLINVAR.exists():
    raise FileNotFoundError(f'File tidak ditemukan: {RAW_CLINVAR}. Pastikan Tahap 1 sudah selesai.')

## 2. FUNGSI BANTU


In [ ]:
# Mengecek apakah file gzip valid dan tidak corrupt
def is_valid_gzip(file_path: Path) -> bool:
    try:
        with gzip.open(file_path, 'rb') as handle:
            handle.read(1)
        return True
    except OSError:
        return False

# Menyeragamkan penulisan label
def normalize_clinical_significance(raw_value: object) -> str | None:
    if pd.isna(raw_value):
        return None

# Membuang label conlict dan none
    text = str(raw_value).strip().lower()
    if not text or text in {'na', 'n/a', '.', 'not provided'}:
        return None
    if 'conflicting' in text or 'risk factor' in text or 'drug response' in text or 'protective' in text:
        return None
    if 'likely pathogenic' in text or ('pathogenic' in text and 'likely' in text):
        return 'Likely pathogenic'
    if text == 'pathogenic' or text.startswith('pathogenic'):
        return 'Pathogenic'
    if 'likely benign' in text or ('benign' in text and 'likely' in text):
        return 'Likely benign'
    if text == 'benign' or text.startswith('benign'):
        return 'Benign'
    if 'uncertain significance' in text or text == 'vus' or 'variant of uncertain significance' in text:
        return 'VUS'

    return None

# mencetak dataframe pada tabel
def print_box_table(preview_df: pd.DataFrame) -> None:
    """Cetak DataFrame kecil sebagai tabel box rapi tanpa pemotongan kolom."""
    if preview_df.empty:
        print('   (tidak ada data untuk ditampilkan)')
        return

    display_df = preview_df.copy().fillna('')
    display_df = display_df.astype(str).map(lambda text: str(text).replace('\n', ' ').replace('\r', ' '))
    headers = [str(column).replace('\n', ' ').replace('\r', ' ') for column in display_df.columns]
    rows = display_df.values.tolist()

    widths = []
    for index, header in enumerate(headers):
        column_cells = [row[index] for row in rows]
        widths.append(max([len(header), *[len(cell) for cell in column_cells], 3]))

    def format_row(values) -> str:
        formatted_cells = []
        for index, value in enumerate(values):
            formatted_cells.append(f' {str(value).ljust(widths[index])} ')
        return '│' + '│'.join(formatted_cells) + '│'

    def border(left: str, middle: str, right: str) -> str:
        return left + middle.join('─' * (width + 2) for width in widths) + right

    top_border = border('┌', '┬', '┐')
    separator = border('├', '┼', '┤')
    bottom_border = border('└', '┴', '┘')

    print(top_border)
    print(format_row(headers))
    print(separator)
    for row in rows:
        print(format_row(row))
    print(bottom_border)

## 3. CHECKPOINT: JIKA CSV BERSIH SUDAH ADA

In [ ]:
#buat ngecek file csv dari clinvar yang sudah ada
if CLEAN_CLINVAR.exists():
    print('\nCheckpoint ditemukan: file ClinVar bersih sudah tersedia.')
    df = pd.read_csv(CLEAN_CLINVAR, dtype=str)
    print(f'Bentuk data tersimpan: {df.shape[0]:,} baris x {df.shape[1]} kolom')
    print('\n20 Data Teratas (Cached Cleaned File):')
    print_box_table(df.head(20))

    top20_path = LAMPIRAN_DIR / 'clinvar_cleaned_grch38_top20.csv'
    df.head(20).to_csv(top20_path, index=False)
    print(f'Top 20 data tersimpan di: {top20_path}')
    print('\nDistribusi kelas:')
    print(df['label'].value_counts())
    print('\nTahap 2 dilewati karena output sudah ada.')
    raise SystemExit(0)

## 4. VALIDASI FILE GZIP


In [ ]:
if not is_valid_gzip(RAW_CLINVAR):
    raise ValueError(
        f'File {RAW_CLINVAR} tidak valid sebagai gzip. Hapus file tersebut lalu jalankan ulang Tahap 1.'
    )

## 5. BACA DAN FILTER DATA


In [ ]:
#mencari kolom utama di data clinvar
print('\nMembaca data mentah ClinVar...')
usecols = [
    'VariationID',
    'Type',
    'ClinicalSignificance',
    'Assembly',
    'Chromosome',
    'PositionVCF',
    'ReferenceAlleleVCF',
    'AlternateAlleleVCF',
]

df = pd.read_csv(RAW_CLINVAR, sep='\t', compression='gzip', usecols=usecols, dtype=str)
print(f'Total data mentah awal: {len(df):,} baris x {df.shape[1]} kolom')

print('\nDistribusi Versi Assembly (Sebelum Difilter):')
print(df['Assembly'].value_counts())

print('\n20 Data Mentah Teratas:')
print_box_table(df.head(20))

#memfilter agar hanya menggunakan versi GRCh38
print('\nMelakukan filtering data...')
df = df[df['Assembly'] == 'GRCh38']
print(f'  Setelah filter Assembly GRCh38: {len(df):,} baris')

#memfilter agar hanya fokus pada SNV
df = df[df['Type'] == 'single nucleotide variant']
print(f'  Setelah filter tipe SNV: {len(df):,} baris')

#menetapkan hanya 4 label yang akan diolah
df['label'] = df['ClinicalSignificance'].apply(normalize_clinical_significance)
allowed_labels = ['Benign', 'Likely benign', 'Likely pathogenic', 'Pathogenic']
df = df[df['label'].isin(allowed_labels)]
print(f'  Setelah filter 4 kelas target (tanpa VUS): {len(df):,} baris')

df = df.dropna(subset=['Chromosome', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'label'])
df = df[(df['PositionVCF'] != 'na') & (df['ReferenceAlleleVCF'] != 'na') & (df['AlternateAlleleVCF'] != 'na')]
print(f'  Setelah menghapus missing values pada kolom kunci: {len(df):,} baris')

#mengecek apakah data memiliki kolom koordinat, jika gaada maka di drop
print('\nMenyimpan data yang sudah bersih...')
df = df.rename(
    columns={
        'PositionVCF': 'pos',
        'Chromosome': 'chr',
        'ReferenceAlleleVCF': 'ref',
        'AlternateAlleleVCF': 'alt',
    }
)
df = df[['VariationID', 'chr', 'pos', 'ref', 'alt', 'label']]
df.to_csv(CLEAN_CLINVAR, index=False)

print(f'Berhasil! Data ClinVar bersih tersimpan di:\n   {CLEAN_CLINVAR}')
print(f'Bentuk data akhir: {df.shape[0]:,} baris x {df.shape[1]} kolom')
print('\n20 Data Teratas (Sudah Dibersihkan):')
print_box_table(df.head(20))

top20_path = LAMPIRAN_DIR / 'clinvar_cleaned_grch38_top20.csv'
df.head(20).to_csv(top20_path, index=False)
print(f'Top 20 data (untuk dosen) tersimpan di:\n   {top20_path}')
print('\nDistribusi Kelas Target:')
print(df['label'].value_counts())


Membaca data mentah ClinVar...
Total data mentah awal: 9,036,351 baris x 8 kolom

Distribusi Versi Assembly (Sebelum Difilter):
Assembly
GRCh37    4537715
GRCh38    4484398
na           9467
NCBI36       4771
Name: count, dtype: int64

20 Data Mentah Teratas:
┌───────────────────────────┬─────────────────────────────────────────────────────┬──────────┬────────────┬─────────────┬─────────────┬────────────────────┬────────────────────────┐
│ Type                      │ ClinicalSignificance                                │ Assembly │ Chromosome │ VariationID │ PositionVCF │ ReferenceAlleleVCF │ AlternateAlleleVCF     │
├───────────────────────────┼─────────────────────────────────────────────────────┼──────────┼────────────┼─────────────┼─────────────┼────────────────────┼────────────────────────┤
│ Indel                     │ Pathogenic/Likely pathogenic                        │ GRCh37   │ 7          │ 2           │ 4820844     │ GGAT               │ TGCTGTAAACTGTAACTGTAAA │
│ Indel    

## 6. BERSIHKAN MEMORI


In [ ]:
del df
gc.collect()
print('\nTAHAP 2 SELESAI!')


TAHAP 2 SELESAI!
